<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Lab_1_Cleaner_Robot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 - Cleaner Robot

In this laboration you will explore knowledge-based AI-agents for the Cleaner Robot domain.

**Do the labs top to bottom.**
<br />
**It is not needed to read further ahead than the current lab section.**

<br />
<hr />

### **Authors**
Mattias Tiger, mattias.tiger@liu.se

### **License**
CC BY-NC-SA 4.0 <br />
https://creativecommons.org/licenses/by-nc-sa/4.0/

The lab (Notebook: code and instructions) is free to use, share, change and to make your own version suitable for your own students. The only restriction to this version and any new version is that 1) the same license (CC BY-NC-SA 4.0) is applied, 2) that the above mentioned authors are referred to as the authors of the original version and 3) that it cannot be used in a setting where a person or business has to pay to get access to, or in order to use, the version. I.e. full permission is given to use the lab, and subsequent versions, in schools or study circles, even running on payed-for computational resources (e.g. payed version of Google colab).
<hr />

# Clearner Robot - Lab 1.0 - The Grid World
We will start by setting up the environment: **The Grid World**.

This include the code to generate a new random **environment** (according to the desired density of walls and dirt) for a given grid size.

We also include the functionality to generate a **scenario** for a given world: Placing the robot at a certain starting point (its home).

With a generated scenario we have a **problem instance**. We can then ask the question: How should the robot move about in the world in order to clean up all the dirt?

### Step 1: Create a problem instance

* **Task 1.0.1:** Make a shallow read-through of the code of all code cells. Focus on the class/function names and descriptions (comments).

  * Question 1: The world in this domain is made up of a grid, where each cell can be of several different types. How many cell types are there and which are they?

* **Task 1.0.2:** Run the code cells.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import random
import math
import copy

# Create output widget for program output
# It is used to capture standard string streams such as print(...)
# for easy debuging and information from inside algorithms
# Usage:
#   with output:
#     print("Hello World")
output = widgets.Output()

In [ ]:
# Functionality to generate and print scenarios
# ---------------------------------------------

# Grid cell types
class CellType:
  EMPTY = 0
  WALL = 1
  DIRT = 2
  HOME = 3
  UNKNOWN = 4


# Convert cell type to string for intelligable printing
def cell_to_string(cell):
  if cell == CellType.EMPTY:
    return "."
  if cell == CellType.WALL:
    return "#"
  if cell == CellType.DIRT:
    return "░"
  if cell == CellType.HOME:
    return "H"
  if cell == CellType.UNKNOWN:
    return "?"
  return f"Error, no cell called {cell}"


def generate_world(grid_size, wall_density, dirt_density):
  # Create grid
  area = (grid_size-2)**2
  grid = np.zeros((grid_size, grid_size), dtype=int)

  # Add surrounding walls
  for row in range(grid_size):
      grid[row, 0] = CellType.WALL  # Wall
      grid[row, grid_size - 1] = CellType.WALL  # Wall
  for col in range(grid_size):
      grid[0, col] = CellType.WALL  # Wall
      grid[grid_size - 1, col] = CellType.WALL  # Wall

  # Add random walls
  num_walls = math.ceil(area*wall_density)
  for _ in range(num_walls):
    tries = 1000
    while grid[row, col] != CellType.EMPTY and tries > 0:
      row = random.randint(1, grid_size - 2)
      col = random.randint(1, grid_size - 2)
      tries -= 1
    if tries > 0:
      grid[row, col] = CellType.WALL  # Wall

  # Add random dirt
  num_dirt = math.ceil(area*dirt_density)
  for _ in range(num_dirt):
    tries = 1000
    while grid[row, col] != CellType.EMPTY and tries > 0:
      row = random.randint(1, grid_size - 2)
      col = random.randint(1, grid_size - 2)
      tries -= 1
    if tries > 0:
      grid[row, col] = CellType.DIRT  # Dirt

  return grid


def generate_scenario(grid_size, wall_density, dirt_density, random_home):
  # Generate world
  grid = generate_world(grid_size, wall_density, dirt_density)

  # Add robot
  robot_pos = (1,1)  # star position at (x,y) coordinte
  grid[robot_pos[0], robot_pos[1]] = CellType.HOME  # Robot home

  return grid, robot_pos


# Helper function to print the world
def print_grid(grid, robot_location_x, robot_location_y, margin=""):
  for y, row in enumerate(grid):
      for x, cell in enumerate(row):
        if x == 0:
          print(margin, end="")
        if robot_location_y == y and robot_location_x == x:
          print("R", end=" ")
        else:
          print(cell_to_string(cell), end=" ")
      print()


# Helper function to print the world and the location coordinates
def print_grid_indices(grid, robot_location_x=1, robot_location_y=1, with_symbol=False):
  for y, row in enumerate(grid):
      for x, cell in enumerate(row):
        if with_symbol:
          if robot_location_y == y and robot_location_x == x:
            symbol = "R"
          else:
            symbol = cell_to_string(cell)
          print(f"({x},{y}, {symbol})", end=" ")
        else:
          print(f"({x},{y})", end=" ")
      print()


# Function to update grid display
def render_grid(render_target, grid, robot_pose):
  with render_target:
    clear_output(wait=True)
    print()
    print_grid(grid, robot_pose[0], robot_pose[1])
    print()




* **Task 1.0.3:** Run the code cell to visualize a generated problem instance.

  * Question 2: What is the size of the grid world generated?
  * Question 3: What happens if you change wall_density (between 0.0 and 1.0)?
  * Question 4: What happens if you change dirt_density (between 0.0 and 1.0)?

(Don't forget to re-run the cells when changing them)
  

In [ ]:
# Create a scenario
# -----------------

# Scenario parameters
grid_size = 10
wall_density = 0.1
dirt_density = 0.1
random_home = False

# Generate scenario
grid, robot_pos = generate_scenario(grid_size, wall_density, dirt_density, random_home)

# Visualize the scenario:
# Create output widget for grid display
output_grid = widgets.Output()
render_grid(output_grid, grid, robot_pos)
print("The grid world (printing the environment):")
display(output_grid)



---



* **Task 1.0.4:** Run the code cell to visualize the locations of each grid cell

  * Question 5: Notice how x increase to the right. How does y change downwards? Is it as expected?

(This is how an AI agent sees the world: (x,y)-coordinate and cell type, for each cell.)

In [ ]:
# Visualize additional info about the scenario
print("The location (x,y) of each grid world cell:")
print_grid_indices(grid)
print()
print("The location of each grid world cell together with its grid cell type (x,y,type):")
print_grid_indices(grid, robot_pos[0], robot_pos[1], with_symbol=True)

# Cleaner Robot - Lab 1.1 - A problem instance simulation
The grid from previously is merely a representation of the state of the world.
We need to add a full environment which an AI-agent can interact with (such that the world, a.k.a. the grid, changes based on agent actions).

To play around with a cleaner robot by hand (and for an AI-agent to control it) we also need some kind of simulation which facilitates the back-and-forth interaction between the robot and the environment.

The simulation will also keep track of the score (we want to reach a good score by cleaning all the dirt and then getting back home, using as few actions as possible).

* **Task 1.1.1:** Make a shallow read-through of the code. Focus on the function names and descriptions.
  * Question 1: Which actions can the Cleaner robot perform?


In [ ]:
# Define all agent actions
AGENT_ACTION_NOOP = 0
AGENT_ACTION_MOVE_LEFT = 1
AGENT_ACTION_MOVE_DOWN = 2
AGENT_ACTION_MOVE_RIGHT = 3
AGENT_ACTION_MOVE_UP = 4
AGENT_ACTION_SUCK_DIRT = 5
AGENT_ACTIONS = [AGENT_ACTION_NOOP, AGENT_ACTION_MOVE_UP, AGENT_ACTION_MOVE_DOWN, AGENT_ACTION_MOVE_LEFT, AGENT_ACTION_MOVE_RIGHT, AGENT_ACTION_SUCK_DIRT]

# The second first to the second last are the move-actions of the agent
AGENT_MOVE_ACTIONS = AGENT_ACTIONS[1:-1]

# Convert action to string for intelligable printing
def action_to_string(action):
  if action == AGENT_ACTION_NOOP:
    return "NOOP"
  if action == AGENT_ACTION_MOVE_UP:
    return "UP"
  if action == AGENT_ACTION_MOVE_DOWN:
    return "DOWN"
  if action == AGENT_ACTION_MOVE_LEFT:
    return "LEFT"
  if action == AGENT_ACTION_MOVE_RIGHT:
    return "RIGHT"
  if action == AGENT_ACTION_SUCK_DIRT:
    return "SUCK"
  return f"Error, no action called {action}"

# ---------------------------------------------------------------------------
# Create an environment class which implements the percepts of a robot of its
# environment and which implements the actions a robot may execute to affect
# the environment.

class Environment:
  def __init__(self, grid_size, wall_density, dirt_density):
    self.world = generate_world(grid_size, wall_density, dirt_density)
    self.agent_location_x = 1
    self.agent_location_y = 1
    self.world[self.agent_location_y][self.agent_location_x] = CellType.HOME
    self.messages = []

  def percept(self, agent):
    return {"home": self.agent_location_x == 1 and self.agent_location_y == 1,
            "dirt": self.world[self.agent_location_y][self.agent_location_x] == CellType.DIRT,
            "bump": agent.bump}

  def is_bump(self, x, y):
    return self.world[y][x] == CellType.WALL

  def execute(self, agent, action):
    agent.bump = False
    agent.state.last_action = action

    x = self.agent_location_x
    y = self.agent_location_y

    # Compute agent performance (for the action)
    if action == AGENT_ACTION_SUCK_DIRT and self.world[y][x] == CellType.DIRT:
        agent.performance += 100.0
        self.messages.append("Performance +100")
    else:
      agent.performance -= 1.0
      self.messages.append("Performance -1")

    # Implement action
    if action == AGENT_ACTION_MOVE_DOWN:
      agent.bump = self.is_bump(x, y + 1)
      if not agent.bump:
        self.agent_location_y += 1
    elif action == AGENT_ACTION_MOVE_UP:
      agent.bump = self.is_bump(x, y - 1)
      if not agent.bump:
        self.agent_location_y -= 1
    elif action == AGENT_ACTION_MOVE_LEFT:
      agent.bump = self.is_bump(x - 1, y)
      if not agent.bump:
        self.agent_location_x -= 1
    elif action == AGENT_ACTION_MOVE_RIGHT:
      agent.bump = self.is_bump(x + 1, y)
      if not agent.bump:
        self.agent_location_x += 1
    elif action == AGENT_ACTION_SUCK_DIRT:
      if self.world[y][x] == CellType.DIRT:
        self.world[y][x] = CellType.EMPTY

  def clear_messages(self):
    self.environment.messages.clear()

# ---------------------------------------------------------------------------
# Create a simple Agent (RemoteCleanerAgent), based on a general Agent class
# which keeps track of the agents knowledge in an agent state (AgentStateBase)

class AgentStateBase:
  def __init__(self):
    self.location_x = 1
    self.location_y = 1
    self.last_action = AGENT_ACTION_NOOP

class Agent:
  def __init__(self):
    self.state = AgentStateBase()
    self.bump = False
    self.performance = 0.0

class RemoteCleanerAgent(Agent):
  def __init__(self, *args):
    super().__init__(*args)

  def execute(self, percept):
    return AGENT_ACTION_NOOP


# ------------------------------------------------------------------------------
# Create a simulation of the scenario such that actions performed by the robot
# affects the environment

class SimulationBase:
  def __init__(self, environment, agent, output = None, environment_output = None, render_world = True):
    self.environment = environment
    self.agent = agent
    self.step_count = 0
    self.render_world = render_world

    self.status_output = output
    self.environment_output = environment_output

    self.agent.performance = -1000
    self.agent.last_action = AGENT_ACTION_NOOP
    self.environment.messages.append(f"action: {action_to_string(self.agent.last_action)}")
    self.environment.messages.append(f"location (environment): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")

  def is_mission_complete(self):
    for y, row in enumerate(self.environment.world):
      for x, cell in enumerate(row):
        if cell == CellType.DIRT:
          return False
    if self.environment.agent_location_x != 1 or self.environment.agent_location_y != 1:
      return False
    return True

  def send_manual_command(self, manual_action):
    with output:

      # Do nothing if all dirt has been collected and the robot is at its home
      if self.is_mission_complete():
        print(f"Mission complete (score: {self.agent.performance})")
        return

      self.step_count += 1
      print(f"{self.step_count} (Iteration)")

      percept = self.environment.percept(self.agent)
      #action = self.agent.execute(percept)
      self.agent.last_action = manual_action
      self.environment.execute(self.agent, manual_action)

      # Set the agent location to the same as the environment location
      # (due to the manual action)
      self.agent.state.location_x = self.environment.agent_location_x
      self.agent.state.location_y = self.environment.agent_location_y


      print(f"  Action: {action_to_string(manual_action)}")
      if manual_action in AGENT_MOVE_ACTIONS:
        if self.agent.bump:
          print(f"  Failed to move {action_to_string(manual_action)}: Wall is in the way.")
        else:
          print(f"  Moved {action_to_string(manual_action)}")
      elif manual_action == AGENT_ACTION_SUCK_DIRT:
        if percept["dirt"]:
          print(f"  Cleaned dirt")
        else:
          print(f"  Failed to clean dirt: No dirt present.")

      self.environment.messages.append(f"manual action: {action_to_string(manual_action)}")
      self.environment.messages.append(f"location (environment): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")
      if self.agent.bump:
        self.environment.messages.append("Bumped")
      else:
        self.environment.messages.append(" ")

      self.render_status()
      if self.environment_output:
        self.render_environment()

      if self.is_mission_complete():
        print(f"Mission complete (score: {self.agent.performance})")


  # Step the simulation and let the agent make decisions on its own
  def step(self, user_feedback=True):
    with output:
      if self.is_mission_complete():
        print(f"Mission complete (score: {self.agent.performance})")
        return

      self.step_count += 1
      print(f"{self.step_count} (Iteration)")

      percept = self.environment.percept(self.agent)
      action = self.agent.execute(percept)
      self.environment.execute(self.agent, action)


      print(f"  Location: {(self.environment.agent_location_x, self.environment.agent_location_y)}   (x,y)")
      print(f"  Action: {action_to_string(action)}")
      if action in AGENT_MOVE_ACTIONS:
        if self.agent.bump:
          print(f"  Failed to move {action_to_string(action)}: Wall is in the way.")
        else:
          print(f"  Moved {action_to_string(action)}")
      elif action == AGENT_ACTION_SUCK_DIRT:
        if percept["dirt"]:
          print(f"  Cleaned dirt")
        else:
          print(f"  Failed to clean dirt: No dirt present.")

      if user_feedback:
        self.environment.messages.append(f"action: {action_to_string(action)}")
        self.environment.messages.append(f"location (environment): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")
        if self.agent.bump:
          self.environment.messages.append("Bumped")
        else:
          self.environment.messages.append(" ")

        self.render()

      if self.is_mission_complete():
        print(f"Mission complete (score: {self.agent.performance})")


  def render_environment(self):
    if self.render_world:
      render_grid(self.environment_output,
                  self.environment.world,
                  (self.environment.agent_location_x, self.environment.agent_location_y))
    else:
      self.environment_output.clear_output()
    with self.environment_output:
      for line in self.environment.messages:
        print(line)
      self.environment.messages.clear()

  def render_status(self):
    if self.status_output:
      self.status_output.clear_output()
      with self.status_output:
        print(f"Iteration: {self.step_count}")
        print(f"Performance: {self.agent.performance}")

  def render(self):
    self.render_status()
    self.render_environment()



* **Task 1.1.2:** Run the code cells (above and below) to start the simulation
  * Question 2: What is the starting score (performance)?
  * Question 3: What happens to the score if you move the robot?

(There are buttons at the bottom of the GUI for you to send commands to the robot.)

* **Task 1.1.3:** Solve three problem instances by controlling the robot with the buttons, cleaning up all dirt and moving back to the robot home position (starting position). A new problem instance is produced by re-running the code cell.
  * Question 4: What score did you get for the three runs? How many actions (i.e. iterations) did it take?
  * Question 5: Did you find any problem instance more difficult than another?

(To finish solving a problem instance, don't forget to get back to the robot home, located at (1,1) and indicated with a 'H' in the world.)

* **Task 1.1.4:** Explore at least three different problem instances where you change the wall density (wall_density) to be higher than default.
  * Question 4: Did it become harder to solve?

* **Task 1.1.5:** Change the wall density (wall_density) to zero.
  * Question 4: If you pretent to be a robot, can you imagine any easy strategy to solve problem instances with no walls?

In [ ]:
# ------------------------------------------------------------------------------
# Setup the GUI (graphical user interface)
# and launch the simulator!

# Create buttons
up_button = widgets.Button(description="Up")
down_button = widgets.Button(description="Down")
left_button = widgets.Button(description="Left")
right_button = widgets.Button(description="Right")
dirt_button = widgets.Button(description="Suck Dirt")

# Create output widget for grid display
environment_world = widgets.Output()

# Create output widget for program status
status = widgets.Output()

# Environment parameters  <- Change these!
grid_size = 10
wall_density = 0.1
dirt_density = 0.1

# Initialize Environment, Agent, and Simulation
environment = Environment(grid_size, wall_density, dirt_density)
agent = RemoteCleanerAgent()
simulation = SimulationBase(environment, agent, output = status, environment_output = environment_world)
simulation.render()

# Clear program output
output.clear_output()

# Button click handlers
up_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_MOVE_UP))
down_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_MOVE_DOWN))
left_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_MOVE_LEFT))
right_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_MOVE_RIGHT))
dirt_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_SUCK_DIRT))

# Setup program output (right half of the GUI)
output_title = widgets.HTML(value="<b>Program Output</b>")
view = widgets.VBox([output])
view.layout.flex_flow = 'column-reverse'
program_output = widgets.VBox([output_title, view])
program_output.layout.max_height = '400px'    # <- Decide height of Output widget
program_output.layout.width = '50%'

# Arrange layout
controls = widgets.HBox([up_button, down_button, left_button, right_button, dirt_button])
display(widgets.HBox([widgets.VBox([status, environment_world, controls]), program_output]))

# Cleaner Robot - Lab 1.2 - Solve the problem in a known environment
We will now construct an AI-agent that can solve any problem instance in this domain if the problem instance is fully visible to the agent at the start.

First, an AI-agent need to know what actions it can take given a specific state of the world. In this domain that is simply equal to the robot location.
> In a more challenging domain, it could also include rotation, or if the robot has an object in its grip (and which object), if the light is turned on and so forth. To crack an egg when making pancakes, it might be necessary to first hold the egg (or similarly), for example.



* **Task 1.2.1:** Run the code cell to show a grid world and to generate all possible (i.e. compatible with the rules of the world) actions given the robot's current location.
  * Question 1: How many neighboring cells can the robot get to from its starting location?
  * Question 2: Which action has the robot take to reach each of these neighboring cells?
  
* **Task 1.2.2:** Change the robot location (new_location) to some other locations.
  * Question 1: Can you find a location where the robot can move in all directions (four neighbors)?
  * Question 2: Can you find a location where the robot can move in three directions (three neighbors)?
  * Question 3: Can you find another location where the robot can move in two directions (two neighbors)?
  * Question 4: Can you find a location where the robot can move only one direction (one neighbor)?
  * Question 5: What happens if you place the robot inside a wall?
  * Question 6: Place the robot at location (1,1) again. Are there any parts of the world that the robot cannot move to, given that it can repeatadly move in its four directions (except through walls)?


In [ ]:
# Initialize Environment (fixed)
grid_size = 10
wall_density = 0.3
dirt_density = 0.02
random.seed(8)
environment = Environment(grid_size, wall_density, dirt_density)

# Find all valid neighbors of a specific world position
# (i.e. those that are not occupied by a wall)
def get_neighbors(pos, world):
  neighbors = []
  #for i, j in [(-1, 0), (0, 1), (1, 0), (0, -1)]:
  for i, j in [(0, -1), (1, 0), (0, 1), (-1, 0)]:
    x = pos[0] + i
    y = pos[1] + j
    if 0 <= x < world.shape[0] and 0 <= y < world.shape[1] and world[y,x] != CellType.WALL:
      neighbors.append((x, y))
  return neighbors

# Example
new_location = (1, 1) # <----- Change these! (x,y)
print_grid(environment.world,
           robot_location_x = new_location[0],
           robot_location_y = new_location[1])
print("")
print(f"Free neighbors to {new_location}: {get_neighbors(new_location, environment.world)}   (x, y)")


One way to solve the problem is to traverse the entire grid world, visiting each location at least once, preferrably with as few actions as possible. Then, if the robot is on top of a grid cell containing dirt, clean it. Finally, get back home (either by going back the same way or by finding a shorter path home).

The search space is a graph, i.e. a location (with/without dirt) is a node where the robot is at, and each valid movement action takes it to another node (another location). We accomplish graph-traversal by depth-first traversal.


* **Task 1.2.3:** Run the code cell.

In [ ]:
# Depth-first traversal
#  Finds a path to visit all accessible nodes from a start node
def dft(start_node, map):
  visited = []
  frontier = [start_node]

  path = []
  while frontier:
    last_node = frontier.pop(-1)
    if last_node not in visited:
      path.append(last_node)
      visited.append(last_node)

      neighbors = get_neighbors(last_node, map)

      for neighbor in neighbors:
        frontier.append(neighbor)

  return path

# ---------------------------

# A representation of a location and what action that brought the robot there
class SearchState:
  def __init__(self, location_x, location_y, parent = None, last_action = None):
    self.location_x = location_x
    self.location_y = location_y
    self.last_action = last_action
    self.parent = parent
  def __hash__(self):
      return hash((self.location_x,self.location_y))

  # This is needed for the visited set
  def __eq__(self, other):
      return self.location_x == other.location_x and self.location_y == other.location_y

# What new state does an action lead to given a current state (search_state)
def apply_action(action, search_state):
  new_state = copy.deepcopy(search_state)
  if action == AGENT_ACTION_MOVE_UP:
    new_state.location_y -= 1
  if action == AGENT_ACTION_MOVE_DOWN:
    new_state.location_y += 1
  if action == AGENT_ACTION_MOVE_LEFT:
    new_state.location_x -= 1
  if action == AGENT_ACTION_MOVE_RIGHT:
    new_state.location_x += 1
  return new_state

# Breadth-first search
#  Finds the shortest path (in actions) from a start state to a state
#  that fulfilles the goal conditions (a function that returns True/False)
def bfs(start_state, world, goal_condition):
  visited = []
  frontier = [start_state]

  plan = []
  while frontier:
    current_state = frontier.pop(0)
    if current_state not in visited:
      visited.append(current_state)

    if goal_condition(current_state, world):
      while current_state.parent:
        plan.insert(0, current_state.last_action)
        current_state = current_state.parent
      return plan

    # Expand the current state
    for action in AGENT_MOVE_ACTIONS:
      new_state = apply_action(action, current_state)
      x = new_state.location_x
      y = new_state.location_y
      if world[y][x] != CellType.WALL and new_state not in visited:
        new_state.parent = current_state
        new_state.last_action = action
        frontier.append(new_state)

  # The search space has been exhausted and no plan has been found
  return []





* **Task 1.2.4:** Run the code cell to create an environment and find a depth-first traversal path.
  * Question 1: Does it cover all accessible locations?
  * Question 2: Where does it end up?
  * Question 3: Can the robot follow the path using its actions (one action per step in the path)?
  
(Suggestion: Use a grid-notebook and draw the depth-first traversal path by hand)

In [ ]:
grid_size = 6
dirt_density = 0.08
environment = Environment(grid_size, 0.1, dirt_density)

print_grid(environment.world, robot_location_x=1, robot_location_y=1)

path = dft((1,1), environment.world)
print("Depth-first traversal path", path)
print("path length: ", len(path)-1) # Don't count the initial position



---


* **Task 1.2.5:** Run the code cell to find a plan (sequence of actions) to execute each piece of the path. Together all plans form one long plan.
  * Question 1: Can the robot follow the path now using its plan?
  * Question 2: How much longer is the plan (in number of actions) compared to the depth-first traversal path (in number of path-coordinates, i.e. (x,y))?
  
(Suggestion: Use a grid-notebook and draw a line that follow the plan)

In [ ]:
plan = []
for n in range(1, len(path)):
  p = bfs(SearchState(path[n-1][0],path[n-1][1]),
          environment.world,
          lambda state,world : state.location_x == path[n][0] and state.location_y == path[n][1])
  plan.extend(p)

print("")
print("Plan to visit all locations in the path above (in order)")
print("--------------------------------------------------------")
print("plan (int): ", plan)
print("plan (str): ", [action_to_string(action) for action in plan])
#print("plan: ",[action_to_symbol(action) for action in plan])
print("plan length: ", len(plan))



---


A better strategy, given that we know (or easily can find out) where the dirt is located, is to find the closest location with dirt and get there using the fewest actions. We use Breadth-first search to find a path to the first dirt-location we find during search.


* **Task 1.2.7:** Run the code cell to find a plan to the closest location with dirt.
  * Question 1: If you follow the plan from the robot starting location (1,1), where do you end up?
  * Question 2: What is the shortest plan to the next dirt location?
  * Question 3: Re-run a few times with different robot locations: Does the plans take the robot to the nearest dirt location?

In [ ]:
robot_location = (1, 1) # <----- Change these! (x,y)

print_grid(environment.world, robot_location_x=robot_location[0], robot_location_y=robot_location[1])
plan = bfs(SearchState(robot_location[0],robot_location[1]),
           environment.world,
           lambda state,world : world[state.location_y][state.location_x] == CellType.DIRT)
print(f"plan = [{[action_to_string(action) for action in plan]}]")



---

Repeating this behavior allow the robot AI-agent to efficiently find all dirt locations. When all dirt is cleaned, it can similarily find the shortest action sequence that takes it back home to finish the problem instance.


* **Task 1.2.8:** Run the code cells below.

In [ ]:
# Cleaner agent with inner state (AgentState) and initially unknown map
# ---------------------------------------------------------------------

class Simulation(SimulationBase):
  def __init__(self, environment, agent, output = None, environment_output = None, agent_world_output = None, render_world = True):
    super().__init__(environment, agent, output, environment_output, render_world)
    self.agent_world_output = agent_world_output
    self.environment.messages.append(f"location (agent):       ({self.agent.state.location_x}, {self.agent.state.location_y})")

  def render_environment(self):
    # Render agent world model
    render_grid(self.agent_world_output,
                self.agent.state.world,
                (self.agent.state.location_x, self.agent.state.location_y))
    # Render enviromnment world
    super().render_environment()

  def send_manual_command(self, manual_action):

    super().send_manual_command(manual_action)
    self.agent.state.last_action = AGENT_ACTION_NOOP
    self.agent.state.update_location(self.environment.percept(self.agent))

    # Perform a dummy no-operation action to update the agent perception
    # on the consequence of its action.
    # This simplify manual control.
    #super().send_manual_command(AGENT_ACTION_NOOP)
    #self.step_count -= 1

    #percept = self.environment.percept(self.agent)
    #self.agent.state.last_action = manual_action
    #self.agent.state.update_location(percept)

    self.environment.messages.append(f"manual action: {action_to_string(manual_action)}")
    self.environment.messages.append(f"location (environment): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")
    self.environment.messages.append(f"location (agent):       ({self.agent.state.location_x}, {self.agent.state.location_y})")
    if self.agent.bump:
      self.environment.messages.append("Bumped")
    else:
      self.environment.messages.append(" ")

    super().render()


class AgentState(AgentStateBase):
  def __init__(self, world_max_width, world_max_height):
    # Agent internal state
    self.location_x = 1
    self.location_y = 1
    self.last_action = AGENT_ACTION_NOOP

    # Initialize agent world model
    self.world_width = world_max_width
    self.world_height = world_max_height
    self.world = [[CellType.UNKNOWN for _ in range(world_max_width)] for _ in range(world_max_height)]
    self.world[1][1] = CellType.HOME

  def update_location(self, percept):
    # Update agent location based on percept
    with output:

      # Update world (with assumed location due to last action)
      if percept["bump"]:
        if self.last_action == AGENT_ACTION_MOVE_UP:
          self.update_world(self.location_x, self.location_y-1, CellType.WALL)
        elif self.last_action == AGENT_ACTION_MOVE_DOWN:
          self.update_world(self.location_x, self.location_y+1, CellType.WALL)
        elif self.last_action == AGENT_ACTION_MOVE_LEFT:
          self.update_world(self.location_x-1, self.location_y, CellType.WALL)
        elif self.last_action == AGENT_ACTION_MOVE_RIGHT:
          self.update_world(self.location_x+1, self.location_y, CellType.WALL)

      # Update location
      if not percept["bump"]:
        if self.last_action == AGENT_ACTION_MOVE_UP:
          self.location_y -= 1
        elif self.last_action == AGENT_ACTION_MOVE_DOWN:
          self.location_y += 1
        elif self.last_action == AGENT_ACTION_MOVE_LEFT:
          self.location_x -= 1
        elif self.last_action == AGENT_ACTION_MOVE_RIGHT:
          self.location_x += 1

      # Update world (based percept and current location)
      if percept["dirt"]:
        if self.last_action == AGENT_ACTION_SUCK_DIRT:
          self.update_world(self.location_x, self.location_y, CellType.EMPTY)
        else:
          self.update_world(self.location_x, self.location_y, CellType.DIRT)
      elif percept["home"]:
        self.update_world(self.location_x, self.location_y, CellType.HOME)
      else:
        self.update_world(self.location_x, self.location_y, CellType.EMPTY)

  def update_world(self, x, y, info):
    self.world[y][x] = info

  def print_world_debug(self):
    print_grid(self.world, self.location_x, self.location_y)
    print()


In [ ]:
class RandomCleanerAgent(Agent):
  def __init__(self, *args):
    super().__init__(*args)
    self.state = AgentState(10, 10)

  def execute(self, percept):
    self.state.update_location(percept)
    return random.choice(AGENT_ACTIONS)

# ------------------------------------------------------------------------------


class BftPlanningCleanerAgent(Agent):
  pass

# ------------------------------------------------------------------------------

class BfsPlanningCleanerAgent(Agent):
  def __init__(self, *args):
    super().__init__(*args)
    self.state = AgentState(10, 10)
    self.plan = []

  def execute(self, percept):
    self.state.update_location(percept)

    with output:

      print(f"  Agent execute")
      print_grid(self.state.world, self.state.location_x, self.state.location_y, margin="    ")
      print(f"    Location: {(self.state.location_x, self.state.location_y)}   (x,y)")
      print(f"    Percept: {percept}")

      # If dirty, clean!
      if percept["dirt"]:
        print(f"    Executing action: {action_to_string(AGENT_ACTION_SUCK_DIRT)}")
        return AGENT_ACTION_SUCK_DIRT

      # Find a path to the nearest dirt
      if not self.plan:
        self.plan = bfs(SearchState(self.state.location_x, self.state.location_y),
                        self.state.world,
                        lambda state,world : world[state.location_y][state.location_x] == CellType.DIRT)
        print(f"    new plan = [{[action_to_string(action) for action in self.plan]}]")

      if not self.plan:
        # All accessible dirt has been cleaned.
        if self.state.world[self.state.location_y][self.state.location_x] != CellType.HOME:
          # Find the way home
          print("    Finding a way home")
          self.plan = bfs(SearchState(self.state.location_x, self.state.location_y),
                          self.state.world,
                          lambda state,world : world[state.location_y][state.location_x] == CellType.HOME)
          print(f"    plan = [{[action_to_string(action) for action in self.plan]}]")
        else:
          # Done with cleaning
          print("    Cleaning done!")
          return AGENT_ACTION_NOOP

      if self.plan:
        # Do first action in the plan (and remove it from the plan)
        action = self.plan.pop(0)
        print(f"    Executing current plan: [{[action_to_string(action) for action in self.plan]}]")
        print(f"    Executing action: {action_to_string(action)}")
        return action

    # We should never get here
    return AGENT_ACTION_NOOP




* **Task 1.2.9:** Run the code cell below to start the GUI.
  * Question 1: Test the Random Agent. Use "Step" to run the simulation step by step (or Step10, Step100 for more steps). Does it seem easy or difficult for the Random Agent to solve the problem?
  * Question 2: Test the BFS Planning Agent. How is this agent better than the Random Agent? What does it do that is more intelligent compared to the Random Agent?

In [ ]:
# ------------------------------------------------------------------------------
# Create the GUI for the simulation program

# Create buttons
step_button = widgets.Button(description="Step")
step_10 = widgets.Button(description="Step10")
step_100 = widgets.Button(description="Step100")
clear = widgets.Button(description="Clear Output")


reset_button = widgets.Button(description="Reset")
new_button = widgets.Button(description="New")

agents = [
        ('Random', 'random'),
        ('BFS Planning', 'bfs planning')
    ]

# Dropdown menu for player1
agent_dropdown = widgets.Dropdown(
    options=agents,
    value='bfs planning', #'bfs planning',
    description='Robot',
)

# Create output widget for grid display
environment_world = widgets.Output()

# Create output widget for agent world display
agent_world = widgets.Output()

# Dummy output
dummy_output = widgets.Output()
with dummy_output:
  print(" ")


# Create output widget for program status
status = widgets.Output()


# Initialize random seed configuration
random_state = random.getstate()

def new_environment():
  # Generate a new random environment
  global random_state
  random.random()
  random_state = random.getstate()
  reset(agent_dropdown.value)

def reset(agent_type):
  # Initialize Environment, Agent, and Simulation
  global environment, agent, simulation
  output.clear_output() # Reset log output (right hand side in GUI)
  random.setstate(random_state)

  # Environment parameters
  grid_size = 6
  dirt_density = 0.1
  wall_density = 0.05

  environment = Environment(grid_size, wall_density, dirt_density)
  if agent_type == 'random':
    agent = RandomCleanerAgent()
  if agent_type == 'bfs planning':
    agent = BfsPlanningCleanerAgent()

  # Known environment
  agent.state.world = np.copy(environment.world)

  simulation = Simulation(environment, agent, output = status, environment_output = environment_world, agent_world_output = agent_world)
  simulation.render()

# Initialize the initial environment, Agent, and Simulation
reset(agent_dropdown.value)


from time import sleep
def step(b, N, increment=1):
  b.disabled = True
  for n in range(N):
    if simulation.is_mission_complete():
      simulation.render()
      break
    if n % increment == 0:
      simulation.step()
      sleep(0.5)
    else:
      simulation.clear_messages()
      simulation.step(user_feedback=False)
  b.disabled = False

# Button click handlers
new_button.on_click(lambda b: new_environment())
reset_button.on_click(lambda b: reset(agent_dropdown.value))
agent_dropdown.observe(lambda b: reset(b['new']) if b['name'] == 'value' and b['type'] == 'change' else None)
step_button.on_click(lambda b: simulation.step())
step_100.on_click(lambda b: step(b, 100, increment=10))
step_10.on_click(lambda b: step(b, 10))
clear.on_click(lambda b: output.clear_output())

# Arrange layout
controls = widgets.HBox([step_button, step_10, step_100, clear])
display(widgets.HBox([widgets.VBox([widgets.HBox([new_button, reset_button, agent_dropdown, dummy_output]), status, widgets.HBox([environment_world, dummy_output, agent_world]), controls]), program_output]))

# Cleaner Robot - Lab 1.3 - Change to an unknown environment

Here, we make the world unknown such that the robot has to explore the world to find where the durt (and walls) are.

* **Task 1.3.1:** Run the code cell below to start the GUI. Solve the problem three times (re-run the code cells)
  * Question 1: Is this problem harder than the one where the world is visible (known) at the start?
  * Question 2: Can such problem instances be solved with our previous efficient solution (BFS to nearest dirty location)?
  * Question 3: What is a good alternative strategy?

In [ ]:
# A remote-controlled agent with its own unknown world model
class CleanerAgent(Agent):
  def __init__(self, *args):
    super().__init__(*args)
    self.state = AgentState(10, 10)

  def execute(self, percept):
    self.state.update_location(percept)
    return AGENT_ACTION_NOOP

# ------------------------------------------------------------------------------

# Create buttons
up_button = widgets.Button(description="Up")
down_button = widgets.Button(description="Down")
left_button = widgets.Button(description="Left")
right_button = widgets.Button(description="Right")
dirt_button = widgets.Button(description="Suck Dirt")

# Create output widget for grid display
environment_world = widgets.Output()

# Create output widget for agent world display
agent_world = widgets.Output()

# Dummy output
dummy_output = widgets.Output()
with dummy_output:
  print(" ")


# Create output widget for program status
status = widgets.Output()

# Initialize Environment, Agent, and Simulation
environment = Environment(grid_size, wall_density, dirt_density)
agent = CleanerAgent()
simulation = Simulation(environment, agent, output = status, environment_output = environment_world, agent_world_output = agent_world, render_world = False)
simulation.render()

# Clear program output
output.clear_output()

# Button click handlers
up_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_MOVE_UP))
down_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_MOVE_DOWN))
left_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_MOVE_LEFT))
right_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_MOVE_RIGHT))
dirt_button.on_click(lambda b: simulation.send_manual_command(AGENT_ACTION_SUCK_DIRT))

# Arrange layout
controls = widgets.HBox([up_button, down_button, left_button, right_button, dirt_button])
#display(widgets.HBox([widgets.VBox([status, widgets.HBox([environment_world, dummy_output, agent_world]), controls]), program_output]))
display(widgets.HBox([widgets.VBox([status, widgets.HBox([agent_world]), controls]), program_output]))

# Cleaner Robot - Lab 1.4 - Solve the problem in an unknown environment

We change BFS to look for the nearest unknown location, as opposed to the nearest unknown cell. Now we have an AI-agent that is once more efficient for this new more difficult domain.

* **Task 1.4.1:** Run the code cell below to start the GUI.
  * Question 1: Use the Random Agent. Use 'Step' to step the simulation once (or use 'Step10' or 'Step100' for faster simulation). Does it seem easy or hard for the Random Agent to solve the problem? Why do you think it is so?
  * Question 2: Use BFS Planning Agent (select it in the drop-down menu). In what ways is this agent better than Random Agent? What makes it more intelligent than Random Agent?
  * Question 3: Try a larger world (e.g. grid_size=10). How many moves does it take for BFS Planning Agent to solve it? Does it seem to be harder for the Random agent?
  * Question 4: Reflect on the difference between this AI-agent and the previous ones.


In [ ]:
class BfsPlanningUnknownCleanerAgent(Agent):
  def __init__(self, *args):
    super().__init__(*args)
    self.state = AgentState(10, 10)
    self.plan = []

  def execute(self, percept):
    self.state.update_location(percept)

    with output:

      print(f"  Agent execute")
      print_grid(self.state.world, self.state.location_x, self.state.location_y, margin="    ")
      print(f"    Location: {(self.state.location_x, self.state.location_y)}   (x,y)")
      print(f"    Percept: {percept}")

      # If dirty, clean!
      if percept["dirt"]:
        print(f"    Executing action: {action_to_string(AGENT_ACTION_SUCK_DIRT)}")
        return AGENT_ACTION_SUCK_DIRT

      # Find a path to the nearest dirt
      # NEW: Look for UNKNOWN to explore instead of DIRT to clean
      if not self.plan:
        self.plan = bfs(SearchState(self.state.location_x, self.state.location_y),
                        self.state.world,
                        lambda state,world : world[state.location_y][state.location_x] == CellType.UNKNOWN)
        print(f"    new plan = [{[action_to_string(action) for action in self.plan]}]")

      if not self.plan:
        # All accessible dirt has been cleaned.
        if self.state.world[self.state.location_y][self.state.location_x] != CellType.HOME:
          # Find the way home
          print("    Finding a way home")
          self.plan = bfs(SearchState(self.state.location_x, self.state.location_y),
                          self.state.world,
                          lambda state,world : world[state.location_y][state.location_x] == CellType.HOME)
          print(f"    plan = [{[action_to_string(action) for action in self.plan]}]")
        else:
          # Done with cleaning
          print("    Cleaning done!")
          return AGENT_ACTION_NOOP

      if self.plan:
        # Do first action in the plan (and remove it from the plan)
        action = self.plan.pop(0)
        print(f"    Executing current plan: [{[action_to_string(action) for action in self.plan]}]")
        print(f"    Executing action: {action_to_string(action)}")
        return action

    # We should never get here
    return AGENT_ACTION_NOOP

# ------------------------------------------------------------------------------

# Create buttons
step_button = widgets.Button(description="Step")
step_10 = widgets.Button(description="Step10")
step_100 = widgets.Button(description="Step100")
clear = widgets.Button(description="Clear Output")


reset_button = widgets.Button(description="Reset")
new_button = widgets.Button(description="New")

agents = [
        ('Random', 'random'),
        ('BFS Planning', 'bfs planning')
    ]

# Dropdown menu for player1
agent_dropdown = widgets.Dropdown(
    options=agents,
    value='random', #'bfs planning',
    description='Robot',
)

# Create output widget for grid display
environment_world = widgets.Output()

# Create output widget for agent world display
agent_world = widgets.Output()

# Dummy output
dummy_output = widgets.Output()
with dummy_output:
  print(" ")


# Create output widget for program status
status = widgets.Output()


# Initialize random seed configuration
random_state = random.getstate()

def new_environment():
  # Generate a new random environment
  global random_state
  random.random()
  random_state = random.getstate()
  reset(agent_dropdown.value)

def reset(agent_type):
  # Initialize Environment, Agent, and Simulation
  global environment, agent, simulation
  output.clear_output() # Reset log output (right hand side in GUI)
  random.setstate(random_state)
  grid_size = 5         # <---------------------------------- Change this value!
  dirt_density = 0.1
  wall_density = 0.05   # <---------------------------------- Change this value!
  environment = Environment(grid_size, wall_density, dirt_density)
  if agent_type == 'random':
    agent = RandomCleanerAgent()
  if agent_type == 'bfs planning':
    agent = BfsPlanningUnknownCleanerAgent()
  simulation = Simulation(environment, agent, output = status, environment_output = environment_world, agent_world_output = agent_world)
  simulation.render()

# Initialize the initial environment, Agent, and Simulation
reset(agent_dropdown.value)


from time import sleep
def step(b, N, increment=1):
  b.disabled = True
  for n in range(N):
    if simulation.is_mission_complete():
      simulation.render()
      break
    if n % increment == 0:
      simulation.step()
      sleep(0.5)
    else:
      simulation.environment.messages = []
      simulation.step(user_feedback=False)
  b.disabled = False

# Button click handlers
new_button.on_click(lambda b: new_environment())
reset_button.on_click(lambda b: reset(agent_dropdown.value))
agent_dropdown.observe(lambda b: reset(b['new']) if b['name'] == 'value' and b['type'] == 'change' else None)
step_button.on_click(lambda b: simulation.step())
step_100.on_click(lambda b: step(b, 100, increment=10))
step_10.on_click(lambda b: step(b, 10))
clear.on_click(lambda b: output.clear_output())

# Arrange layout
controls = widgets.HBox([step_button, step_10, step_100, clear])
display(widgets.HBox([widgets.VBox([widgets.HBox([new_button, reset_button, agent_dropdown, dummy_output]), status, widgets.HBox([environment_world, dummy_output, agent_world]), controls]), program_output]))